<a href="https://colab.research.google.com/github/Rogendo/Machine-Learning-models/blob/main/JengaNLP_Official_Quickstart_Annotated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 JengaNLP Quickstart

Welcome to the official JengaNLP onboarding notebook.

**New to JengaNLP?** This notebook is a runnable, annotated version of the official docs: usage guide,  exact workflow, YAML config reference, and CLI vs. Python vs. production inference. Each section below explains *what* a cell does and *which* doc section it comes from, so you don't need to have read the docs first.

## What you'll build
- Create a JSONL dataset
- Configure an experiment using YAML
- Train a transformer model
- Evaluate it
- Run inference from the CLI and Python

## Framework Flow

```text
Dataset
   ↓
YAML Configuration
   ↓
ExperimentConfig
   ↓
DataProcessor
   ↓
TaskRegistry
   ↓
MultiTaskModel
   ↓
Trainer
   ↓
InferencePipeline
```

This is the same pipeline show's "How Config Maps To Core Modules" — one YAML file drives every stage.

> 💡 This notebook uses `hf-internal-testing/tiny-random-bert` for speed... actually it uses `distilbert-base-uncased` (see the configs below), but the same idea applies: swap `model.base_model` for a bigger MLM (Masked Language Modeling) model such as `bert-base-multilingual-cased` when you move from a smoke test to a real project.


## Step 1: Environment Setup

Colab occasionally ships a `torchvision` build that raises `ImportError: cannot import name 'VideoReader'` when `transformers` is imported. Uninstalling it here avoids that error later (see the **Common troubleshooting** section at the end of this notebook). We also upgrade `pip` so the next install step resolves dependencies cleanly.


In [1]:
!pip uninstall torchvision

Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/torchvision-0.26.0+cpu.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libjpeg.b1a87dc0.so.8
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libpng16.60f56faa.so.16
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libsharpyuv.691ead6e.so.0
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libwebp.e20c0bc7.so.7
    /usr/local/lib/python3.12/dist-packages/torchvision.libs/libz.7dbeb23c.so.1
    /usr/local/lib/python3.12/dist-packages/torchvision/*
Proceed (Y/n)? y
  Successfully uninstalled torchvision-0.26.0+cpu


In [2]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


## Step 2: Install JengaNLP

This installs `jenga-nlp` from **Test PyPI**, a pre-release channel used for early builds. For day-to-day project work, the Docs Installation section recommends one of:

```bash
# Recommended: from PyPI
pip install jenga-nlp

# From source (development / editable install)
git clone https://github.com/Rogendo/JengaNLP.git
cd JengaNLP
pip install -e ".[dev]"
```

The project also pins `setuptools<81` because TensorBoard imports `pkg_resources` in some environments — if you hit a `pkg_resources` error later, run `pip install "setuptools>=68,<81"`.


In [3]:
!pip install --no-deps -i https://test.pypi.org/simple/ jenga-nlp==2.0.0

Looking in indexes: https://test.pypi.org/simple/


## Step 3 (Optional): Experiment Tracking with MLflow

JengaNLP can log training metrics to MLflow via `training.logging` in the YAML config. MLflow itself doesn't need a running server for file-based tracking, but this notebook spins up a local MLflow UI backed by SQLite so you can watch metrics live while training runs later in the notebook.

This whole MLflow/ngrok block is **optional** — skip it and set `training.logging.service: none` in your config if you just want to run the training/inference workflow without a dashboard.


In [4]:
!pip install mlflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 84.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 93.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 48.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 119.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 969.1/969.1 kB 35.8 MB/s  0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 49.0.0
    Uninstalling cryptography-49.0.0:
      Successfully uninstalled cryptography-49.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [mlflow]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jenga-nlp 2.0.0 requires bitsandbytes>=0.43.0, which is not installed.
pyopenssl 26.3.0 requires cryptography<50,>=49.0.0, but you have cryptography 48.0.1 which is incompatible.


In [5]:
# ! mlflow ui  --backend-store-uri sqlite:///mlflow.db  --default-artifact-root ./mlartifacts   --host 0.0.0.0  --port 5000

The cell below launches `mlflow ui` as a **background process** (instead of the commented-out foreground command above) so the notebook can keep running while the MLflow server stays up, backed by `mlflow.db` (SQLite) with artifacts written to `./mlartifacts`.


In [6]:
import subprocess

# Your exact command broken down into a list arguments for background execution
mlflow_cmd = [
    "mlflow", "ui",
    "--backend-store-uri", "sqlite:///mlflow.db",
    "--default-artifact-root", "./mlartifacts",
    "--host", "0.0.0.0",
    "--port", "5000" #,
    # "--allowed-hosts", "*"
]

# Run the server in the background
mlflow_process = subprocess.Popen(mlflow_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("MLflow Tracking Server started successfully with SQLite!")


MLflow Tracking Server started successfully with SQLite!


### Optional: Expose the MLflow UI via ngrok (Colab only)

Colab VMs aren't reachable from your browser directly, so `ngrok` opens a public tunnel to the local MLflow server (port `5000`) so you can view the dashboard. This step needs an `NGROK_TOKEN` stored in Colab Secrets and is **not required** to train or run inference — it only affects whether you can watch metrics in a browser UI.


In [7]:
!pip install pyngrok -q

In [8]:
from google.colab import userdata
from pyngrok import ngrok

# Fetch token from Colab Secrets
ngrok_token = userdata.get('NGROK_TOKEN')

ngrok.set_auth_token(ngrok_token)

In [9]:

# Open a public HTTP tunnel
public_url = ngrok.connect(5000, host_header="localhost:5000")
print("MLflow UI Public URL:", public_url.public_url)



MLflow UI Public URL: https://29c6-34-87-69-104.ngrok-free.app


> ⚠️ **Heads up:** the ngrok URL above is randomly generated and changes every time the tunnel restarts. The fixture configs later in this notebook (`fixture_mpesa_fraud_cpu.yaml`, `fixture_multitask_cpu.yaml`) hard-code a `tracking_uri` value from a *previous* run of this notebook — if you re-run everything from scratch, update `training.logging.tracking_uri` in those configs to match your new ngrok URL (or point it at `sqlite:///mlflow.db` instead, as shown below, to avoid the dependency on ngrok entirely).


 ## **Log a Test Run (Optional)**
 To verify everything works, initialize an MLflow experiment and log some mock metrics

In [10]:
import mlflow

# Point MLflow to the local server
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Colab_Demo")

with mlflow.start_run():
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_metric("accuracy", 0.95)
    print("Logged metrics successfully! Refresh your ngrok URL to view them.")


2026/07/25 08:46:48 INFO mlflow.tracking.fluent: Experiment with name 'Colab_Demo' does not exist. Creating a new experiment.


Logged metrics successfully! Refresh your ngrok URL to view them.
🏃 View run nebulous-steed-207 at: http://127.0.0.1:5000/#/experiments/1/runs/d058d60ef8c54e1391d2829af852cb79
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


The cell below shows the **simpler alternative**: point MLflow directly at a local SQLite file (`sqlite:///mlflow.db`) instead of the ngrok HTTP URL. Use this pattern for local/offline logging when you don't need a shared dashboard — it's the same idea as `tracking_uri: file:///tmp/jenganlp_mlruns` in `configuration.md`'s Logging example.


In [11]:
import mlflow

# Set the tracking URI to your local SQLite database file
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("My_SQLite_Experiment")


2026/07/25 08:46:56 INFO mlflow.tracking.fluent: Experiment with name 'My_SQLite_Experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/mlruns/2', creation_time=1784969216984, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1784969216984, lifecycle_stage='active', name='My_SQLite_Experiment', tags={}, trace_location=None, workspace='default'>

## Step 4: Verify the CLI Installation

Running the module with no subcommand prints usage help — this is the quickest way to confirm `jenga_nlp` installed correctly and to see the available commands (`train`, `evaluate`, `predict`, `resume`, `backends`, `asr`), matching the Docs's **Verify installation** step.


In [12]:
!python -m jenga_nlp

JengaNLP command line

Usage:
  python -m jenga_nlp <command> [options]

Common commands:
  python -m jenga_nlp train --config configs/fixture_mpesa_fraud_cpu.yaml --device cpu
  python -m jenga_nlp evaluate --config configs/fixture_mpesa_fraud_cpu.yaml --model-dir /tmp/jenganlp_fixture_mpesa_fraud_cpu --device cpu
  python -m jenga_nlp predict --config configs/fixture_mpesa_fraud_cpu.yaml --model-dir /tmp/jenganlp_fixture_mpesa_fraud_cpu --text "Your text here"
  python -m jenga_nlp resume --config configs/fixture_mpesa_fraud_cpu.yaml --run-dir /tmp/jenganlp_fixture_mpesa_fraud_cpu
  python -m jenga_nlp backends status --config configs/fixture_mpesa_fraud_cpu.yaml
  python -m jenga_nlp asr serve --model-name openai/whisper-small --audio sample.wav

Commands:
  train       Train a text model from a YAML config.
  evaluate    Evaluate a saved model directory.
  predict     Run inference with a saved model directory.
  resume      Resume training from a saved run directory.
  backends   

## Step 5: Inspect the Training Fixtures

The three JSONL files below already exist in this Colab session's `/content` working directory. Each one is training data for a different **task type**, per `configuration.md`'s Task Types table:

| File | Task type | Label shape |
|---|---|---|
| `sample_classification.jsonl` | `single_label_classification` (fraud detection) | integer class id (`0` = legit, `1` = fraud) |
| `sample_sentiment.jsonl` | `sentiment` | integer class id (`0`=negative, `1`=neutral, `2`=positive) |
| `sample_qa_transcripts.jsonl` | `qa_scoring` | dict of per-criterion binary checklist vectors (`opening`, `listening`, `proactiveness`, `resolution`, `hold`, `closing`) |

**If these files are missing** (e.g. you're running this notebook in a fresh environment), recreate the fraud-detection dataset with the exact snippet from `quickstart.md`:

```bash
mkdir -p /tmp/jenganlp_demo
cat > /tmp/jenganlp_demo/messages.jsonl <<'JSONL'
{"text":"Urgent M-PESA reversal request. Share your PIN now.","label":1}
{"text":"Your payment has been received successfully.","label":0}
{"text":"Congratulations, send your PIN to claim the prize.","label":1}
{"text":"Please visit the nearest branch for help.","label":0}
{"text":"Account blocked. Reply with your secret code.","label":1}
{"text":"Your statement is ready for download.","label":0}
JSONL
```

Supported dataset file formats are `.jsonl`, `.json`, and `.csv`.


In [23]:
!cat /content/sample_qa_transcripts.jsonl

{"transcript": "Good morning, thank you for calling Safaricom customer care, my name is Grace. How may I assist you today? I understand you are having trouble with your M-PESA transfer. Let me look into that for you right away. I can see the transaction was delayed due to network issues. I have initiated a reversal which should reflect in 24 hours. Is there anything else I can help you with? Thank you for calling and have a wonderful day.", "labels": {"opening": [1], "listening": [1, 1, 1, 1, 0], "proactiveness": [1, 1, 1], "resolution": [1, 1, 1, 1, 0], "hold": [0, 1], "closing": [1]}}
{"transcript": "Hello. What do you want? Your M-PESA is not working? Just try again later. I cannot do anything about that. Bye.", "labels": {"opening": [0], "listening": [0, 0, 0, 0, 0], "proactiveness": [0, 0, 0], "resolution": [0, 0, 0, 0, 0], "hold": [1, 1], "closing": [0]}}
{"transcript": "Welcome to customer support. I hear that your account balance is showing incorrectly. That must be frustrating

In [24]:
!cat /content/sample_classification.jsonl

{"text": "Received M-PESA confirmation for KSh 50000 sent to unknown number at 3AM", "label": 1}
{"text": "Payment of KSh 200 for airtime top-up via M-PESA", "label": 0}
{"text": "Multiple rapid transactions totaling KSh 500000 within 10 minutes to different accounts", "label": 1}
{"text": "Salary deposit of KSh 45000 received from employer account", "label": 0}
{"text": "Urgent request to send KSh 100000 to reverse a wrong transaction", "label": 1}
{"text": "Monthly rent payment of KSh 25000 via Lipa Na M-PESA", "label": 0}
{"text": "Account shows withdrawal of KSh 200000 but customer denies making the transaction", "label": 1}
{"text": "School fees payment of KSh 35000 to registered institution", "label": 0}
{"text": "SIM swap followed by immediate large transfer of KSh 150000", "label": 1}
{"text": "Utility bill payment of KSh 3500 for Kenya Power", "label": 0}
{"text": "Customer reports receiving phishing SMS about M-PESA account verification", "label": 1}
{"text": "Regular grocery

In [25]:
!cat /content/sample_sentiment.jsonl

{"text": "M-PESA service has been excellent today, transactions going through instantly", "label": 2}
{"text": "Very frustrated with the slow customer support response times", "label": 0}
{"text": "Account balance updated correctly after the reversal", "label": 1}
{"text": "This is the worst experience I have ever had with mobile money", "label": 0}
{"text": "The new M-PESA app interface is clean and easy to use", "label": 2}
{"text": "I lost money due to a failed transaction and nobody is helping", "label": 0}
{"text": "Transaction completed successfully, nothing special", "label": 1}
{"text": "Absolutely love the instant settlement feature for business payments", "label": 2}
{"text": "System was down for three hours during peak business time", "label": 0}
{"text": "The charges seem reasonable for the service provided", "label": 1}
{"text": "Finally resolved my issue after calling support five times", "label": 1}
{"text": "Outstanding fraud detection caught a suspicious withdrawal imm

## Step 6: Understand the Multi-Task Config

`fixture_multitask_cpu.yaml` trains **one shared encoder** (`distilbert-base-uncased`) with **three task heads** attached — this is what "multi-task" means in JengaNLP. Core Sections table:

- `project_name` — human-readable run name, used in logs and MLflow.
- `model.base_model` — the shared Hugging Face encoder every task sits on top of.
- `tokenizer` — shared `max_length`/padding/truncation settings for all tasks.
- `tasks` — a **list**: `mpesa_fraud_detection` (single-label), `mpesa_sentiment`   (sentiment), and `call_qa_scoring` (qa_scoring with 6 weighted heads —   `weight` controls how much each head contributes to the combined loss).
- `training` — optimizer, batch size, epochs, `device: cpu`, plus the   `logging.tracking_uri` pointing at the MLflow/ngrok URL from Step 3.
- `training.checkpoint` — saves a checkpoint every epoch and keeps the best one.
- `training.data` — the 80/20-ish train/eval split (`test_size`) and random `seed`.

If you only need one task, you don't need `qa_scoring`'s complexity — see Step 8 below for the simpler single-task config this notebook actually trains, evaluates, and serves predictions from.


In [27]:
!cat /content/fixture_multitask_cpu.yaml

project_name: fixture_multitask_cpu

model:
  base_model: distilbert-base-uncased
  dropout: 0.1

tokenizer:
  max_length: 256
  padding: max_length
  truncation: true

tasks:
  - name: mpesa_fraud_detection
    type: single_label_classification
    data_path: /home/naynek/jenga_nlp/examples/fixtures/sample_classification.jsonl
    text_column: text
    label_column: label
    heads:
      - name: fraud_head
        num_labels: 2

  - name: mpesa_sentiment
    type: sentiment
    data_path: /home/naynek/jenga_nlp/examples/fixtures/sample_sentiment.jsonl
    text_column: text
    label_column: label
    heads:
      - name: sentiment_head
        num_labels: 3

  - name: call_qa_scoring
    type: qa_scoring
    data_path: /home/naynek/jenga_nlp/examples/fixtures/sample_qa_transcripts.jsonl
    text_column: transcript
    label_column: labels
    heads:
      - name: opening
        num_labels: 1
        weight: 1.0
      - name: listening
        num_labels: 5
        weight: 1.5
      

## Step 7: Train the Multi-Task Model

This runs the full pipeline described in the Docs "How Config Maps To Core Modules" section:

```text
YAML config → ExperimentConfig → AutoTokenizer → DataProcessor →
TaskRegistry → MultiTaskModel → Trainer → checkpoints/logs saved to output_dir
```

With only a handful of examples per task, CPU training still takes a few minutes because it runs 4 epochs across 3 tasks. Watch the log for a line per epoch showing metrics **per task per head** (e.g. `mpesa_fraud_detection_fraud_head_f1`) — this is normal for multi-task runs and is how you check whether every head is learning, not just the overall loss.


In [40]:
!python -m jenga_nlp train  --config /content/fixture_multitask_cpu.yaml --device cpu

2026-07-25 08:58:30,692 | INFO | jenga_nlp.activity | [system] info run started: fixture_multitask_cpu
2026-07-25 08:58:30,693 | INFO | jenga_nlp.activity | [system] info loading tokenizer
2026-07-25 08:58:32,312 | INFO | jenga_nlp.data.processor | Processing data for task: mpesa_fraud_detection (type: TaskType.SINGLE_LABEL_CLASSIFICATION)
2026-07-25 08:58:32,316 | INFO | jenga_nlp.data.processor |   Loaded 20 rows from sample_classification.jsonl
Map: 100% 20/20 [00:00<00:00, 2517.44 examples/s]
2026-07-25 08:58:32,449 | INFO | jenga_nlp.data.processor |   Task 'mpesa_fraud_detection': 16 train, 4 eval samples
2026-07-25 08:58:32,449 | INFO | jenga_nlp.data.processor | Processing data for task: mpesa_sentiment (type: TaskType.SENTIMENT)
2026-07-25 08:58:32,452 | INFO | jenga_nlp.data.processor |   Loaded 15 rows from sample_sentiment.jsonl
Map: 100% 15/15 [00:00<00:00, 1920.53 examples/s]
2026-07-25 08:58:32,581 | INFO | jenga_nlp.data.processor |   Task 'mpesa_sentiment': 12 train, 3

## Step 8: A Simpler Single-Task Config

`fixture_mpesa_fraud_cpu.yaml` trains **only** the fraud-detection task. This is exactly the config built in `quickstart docs`'s "Create A Config" step, and the rest of this notebook (evaluate → predict → Python inference) follows that guide precisely because a single task is the clearest way to see the full train → evaluate → predict flow without the multi-task noise above.


In [41]:
!cat /content/fixture_mpesa_fraud_cpu.yaml

project_name: fixture_mpesa_fraud_cpu

model:
  base_model: distilbert-base-uncased
  dropout: 0.1

tokenizer:
  max_length: 64
  padding: max_length
  truncation: true

tasks:
  - name: mpesa_fraud_detection
    type: single_label_classification
    data_path: /content/sample_classification.jsonl
    text_column: text
    label_column: label
    heads:
      - name: fraud_head
        num_labels: 2

training:
  output_dir: /content/jenganlp_fixture_mpesa_fraud_cpu
  learning_rate: 0.00005
  batch_size: 2
  eval_batch_size: 2
  num_epochs: 3
  weight_decay: 0.0
  warmup_steps: 0
  max_grad_norm: 1.0
  gradient_accumulation_steps: 1
  use_amp: false
  device: cpu
  metric_for_best_model: eval_loss
  greater_is_better: false
  logging:
    service: mlflow
    experiment_name: fixture_mpesa_fraud_cpu
    tracking_uri: https://29c6-34-87-69-104.ngrok-free.app
    log_every_n_steps: 1
  checkpoint:
    save_every_n_epochs: 1
    save_best: true
    max_checkpoints: 1
  data:
    test_size: 

## Step 9: Train the Single-Task Model

Same `train` command as Step 7, just pointed at the simpler config. Because there's only one task and one head, training finishes faster and the per-epoch metrics (`accuracy`, `precision`, `recall`, `f1`) are easier to read at a glance. The model and checkpoints land in `training.output_dir` (`/content/jenganlp_fixture_mpesa_fraud_cpu`), which the Evaluate/Predict steps below reuse via `--model-dir`.


In [42]:
!python -m jenga_nlp train  --config /content/fixture_mpesa_fraud_cpu.yaml --device cpu

2026-07-25 09:00:38,080 | INFO | jenga_nlp.activity | [system] info run started: fixture_mpesa_fraud_cpu
2026-07-25 09:00:38,080 | INFO | jenga_nlp.activity | [system] info loading tokenizer
2026-07-25 09:00:39,618 | INFO | jenga_nlp.data.processor | Processing data for task: mpesa_fraud_detection (type: TaskType.SINGLE_LABEL_CLASSIFICATION)
2026-07-25 09:00:39,623 | INFO | jenga_nlp.data.processor |   Loaded 20 rows from sample_classification.jsonl
Map: 100% 20/20 [00:00<00:00, 3069.83 examples/s]
2026-07-25 09:00:39,738 | INFO | jenga_nlp.data.processor |   Task 'mpesa_fraud_detection': 16 train, 4 eval samples
2026-07-25 09:00:39,738 | INFO | jenga_nlp.activity | [system] info datasets processed
2026-07-25 09:00:39,993 | INFO | jenga_nlp.tasks.registry | Creating task 'mpesa_fraud_detection' of type 'single_label_classification' with hidden_size=768
Loading weights: 100% 100/100 [00:00<00:00, 298.68it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key   

## Step 10: Evaluate the Model

`evaluate` reloads the saved model directory and reports metrics on the held-out eval split (per `quickstart docs` and `inference docs`). Use this after training to confirm quality before you trust any predictions — there's no point running inference on a model you haven't checked.


In [43]:
!python -m jenga_nlp evaluate \
  --config /content/fixture_mpesa_fraud_cpu.yaml \
  --model-dir /content/jenganlp_fixture_mpesa_fraud_cpu \
  --device cpu


2026-07-25 09:04:47,288 | INFO | jenga_nlp.activity | [system] info run started: fixture_mpesa_fraud_cpu-eval
2026-07-25 09:04:48,869 | INFO | jenga_nlp.data.processor | Processing data for task: mpesa_fraud_detection (type: TaskType.SINGLE_LABEL_CLASSIFICATION)
2026-07-25 09:04:48,874 | INFO | jenga_nlp.data.processor |   Loaded 20 rows from sample_classification.jsonl
Map: 100% 20/20 [00:00<00:00, 2984.10 examples/s]
2026-07-25 09:04:48,990 | INFO | jenga_nlp.data.processor |   Task 'mpesa_fraud_detection': 16 train, 4 eval samples
2026-07-25 09:04:48,990 | INFO | jenga_nlp.activity | [system] info evaluation datasets processed
2026-07-25 09:04:48,990 | INFO | jenga_nlp.tasks.registry | Creating task 'mpesa_fraud_detection' of type 'single_label_classification' with hidden_size=768
Loading weights: 100% 100/100 [00:00<00:00, 8742.87it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+-------

## Step 11: Predict via CLI (Smoke Test)

This is the `inference documentation` of the "CLI Smoke Test" pattern: it's the fastest way to sanity-check a saved model on one piece of text. **Important:** the CLI reloads the entire model from disk on every call — great for a quick check, but not how you'd serve real traffic. For that, see Step 12.


In [44]:
!python -m jenga_nlp predict \
  --config /content/fixture_mpesa_fraud_cpu.yaml \
  --model-dir /content/jenganlp_fixture_mpesa_fraud_cpu \
  --task-name mpesa_fraud_detection \
  --text "Urgent M-PESA reversal request asking for PIN"


2026-07-25 09:05:03,079 | INFO | jenga_nlp.inference.pipeline | Using saved model config: /content/jenganlp_fixture_mpesa_fraud_cpu/experiment_config.yaml
2026-07-25 09:05:04,642 | INFO | jenga_nlp.tasks.registry | Creating task 'mpesa_fraud_detection' of type 'single_label_classification' with hidden_size=768
Loading weights: 100% 100/100 [00:00<00:00, 7392.67it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-07-25 09:05:05,698 | INFO | jenga_nlp.core.model | Model loaded from /content/jenganlp_fixture_mpesa_fraud_cpu
{'text': 'Urgent M-P

## Step 12: Load Once in Python (Production-Style Inference)

This is the `InferencePipeline` pattern from `inference docs` ("Python Load-Once Inference") and `quickstart doc`'s final step — load the model **once**, then call `.predict()` / `.predict_batch()` as many times as you want without reloading. Use this pattern for:

- repeated predictions inside a notebook,
- batch-scoring jobs, and
- production services (e.g. wrapped in a FastAPI app — see `inference doc`'s   "Production HTTP Wrapper" section for that example; JengaNLP doesn't   yet ship its own `serve` command).

Note: JengaNLP-native multi-task models like this one always need `jenga_nlp` at inference time, because the task heads and output formatters are custom. Only *standard* exported models (e.g. Whisper, plain `AutoModelForSequenceClassification`) can be loaded with plain `transformers` instead.


In [45]:
from jenga_nlp.inference.pipeline import InferencePipeline

pipeline = InferencePipeline.from_checkpoint(
    model_dir="/content/jenganlp_fixture_mpesa_fraud_cpu",
    config_path="/content/fixture_mpesa_fraud_cpu.yaml",
    device="cpu",
)

result = pipeline.predict(
    text="Urgent M-PESA reversal request asking for PIN",
    task_name="mpesa_fraud_detection",
)
print(result.to_dict())

# Batch prediction: score many texts in one call instead of one at a time
batch_results = pipeline.predict_batch(
    texts=[
        "Urgent M-PESA reversal request asking for PIN",
        "Your payment has been received successfully.",
    ],
    task_name="mpesa_fraud_detection",
    batch_size=16,
)
for item in batch_results:
    print(item.to_dict())


2026-07-25 09:05:08,339 | INFO | jenga_nlp.inference.pipeline | Using saved model config: /content/jenganlp_fixture_mpesa_fraud_cpu/experiment_config.yaml


INFO:jenga_nlp.inference.pipeline:Using saved model config: /content/jenganlp_fixture_mpesa_fraud_cpu/experiment_config.yaml


2026-07-25 09:05:09,883 | INFO | jenga_nlp.tasks.registry | Creating task 'mpesa_fraud_detection' of type 'single_label_classification' with hidden_size=768


INFO:jenga_nlp.tasks.registry:Creating task 'mpesa_fraud_detection' of type 'single_label_classification' with hidden_size=768


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-07-25 09:05:10,807 | INFO | jenga_nlp.core.model | Model loaded from /content/jenganlp_fixture_mpesa_fraud_cpu


INFO:jenga_nlp.core.model:Model loaded from /content/jenganlp_fixture_mpesa_fraud_cpu


{'text': 'Urgent M-PESA reversal request asking for PIN', 'task_results': {'mpesa_fraud_detection': {'task_name': 'mpesa_fraud_detection', 'task_type': 'single_label_classification', 'heads': {'fraud_head': {'head_name': 'fraud_head', 'prediction': 1, 'confidence': 0.8749, 'probabilities': {'0': 0.1251, '1': 0.8749}}}}}, 'pii_detected': False, 'processing_time_ms': 99.17}
{'text': 'Urgent M-PESA reversal request asking for PIN', 'task_results': {'mpesa_fraud_detection': {'task_name': 'mpesa_fraud_detection', 'task_type': 'single_label_classification', 'heads': {'fraud_head': {'head_name': 'fraud_head', 'prediction': 1, 'confidence': 0.8749, 'probabilities': {'0': 0.1251, '1': 0.8749}}}}}, 'pii_detected': False, 'processing_time_ms': 90.38}
{'text': 'Your payment has been received successfully.', 'task_results': {'mpesa_fraud_detection': {'task_name': 'mpesa_fraud_detection', 'task_type': 'single_label_classification', 'heads': {'fraud_head': {'head_name': 'fraud_head', 'prediction': 1,

# 🎉 Congratulations

You have completed the JengaNLP Quickstart: dataset → config → train → evaluate → predict (CLI) → predict (Python, load-once).

## Where to go next

- **Multi-task learning** — you already trained one above (Step 7); see   `configuration docs` for `task_sampling` strategies (`round_robin`,   `proportional`, `temperature`).
- **Named Entity Recognition** — `type: ner`, label shape is a list of   entity spans.
- **Regression** — `type: regression`; use `activation: sigmoid` with   `output_min`/`output_max` for bounded scores like a 0.0–1.0 risk score.
- **QA Scoring** — `type: qa_scoring` (seen above in the multi-task   config); this is checklist/compliance scoring, *not* extractive QA —   `question_answering` is a backward-compatible alias for it.
- **Whisper ASR (Experimental)** — a standard Hugging Face architecture   under the hood, so exported Whisper models don't need `jenga_nlp` for   inference; see the Doc's Whisper Workflows section.
- **Production deployment with FastAPI** — wrap `InferencePipeline`   (Step 12 above) in a FastAPI app and load the model once at process   startup, as shown in `inference doc`'s Production HTTP Wrapper example.

## Common troubleshooting

### `ImportError: cannot import name 'VideoReader'`

Cause: incompatible torchvision installation.

Fix:

```bash
pip uninstall torchvision
```

### Hugging Face authentication warning

Set:

```bash
export HF_TOKEN=<your_token>
```

for faster downloads and higher rate limits.

### `python -m jenga_nlp` says a command is required

Run `python -m jenga_nlp -h`, or use one of `train` / `evaluate` / `predict` / `resume` / `backends` / `asr` explicitly.

### `--model-dir` path errors

The path must be a single argument with no space in it, e.g. `--model-dir /content/jenganlp_fixture_mpesa_fraud_cpu` — not `/content/ jenganlp_fixture_mpesa_fraud_cpu`.

### `ModuleNotFoundError: No module named 'pkg_resources'` (TensorBoard)

```bash
pip install "setuptools>=68,<81"
```

### Training finished suspiciously fast, or predictions look wrong

If you swapped in `hf-internal-testing/tiny-random-bert`, that model is intentionally tiny and random — it proves the pipeline mechanics work, not prediction quality. For real quality, use `distilbert-base-uncased` (as this notebook does) or a multilingual/domain model such as `bert-base-multilingual-cased` or `xlm-roberta-base`, with more data and enough epochs.
